# Making peaks coincide on purpose

Two districts share one grid connection. It has to carry their combined load in every hour,
so a model that sizes it needs the **combined peak**. If the typical days underestimate it,
the connection comes out too small.

In this tutorial we reduce two weeks of load in both districts to two typical days and
compare their combined peak with the real one. By the end you will have seen that

* the default ordering underestimates the combined peak, because it keeps the two districts'
  peaks at different hours,
* a shared ordering together with `preserve_minmax=True` gives typical days whose combined
  peak is never lower than the real one, and
* `reference` lets you choose the hour at which that peak occurs.

This is part 5 of five [duration-curve tutorials](../index.md#duration-curves). Each part stands
on its own.

## 1  Two districts, one connection

The example data contains the hourly load of one district. We use it as **district A** and
construct **district B** from it: a smaller district, 60% of the size of A, whose day runs
two hours later. We take **1–14 April 2010**. Two typical days for two weeks is a
deliberately tight budget; it makes the differences easy to see.

Every run in this tutorial uses the same settings, so we write them down once in a small
function. Only its `representation` argument will change. `preserve_column_means=False`
switches off tsam's rescaling step, so that we see the values exactly as the representation
produced them.

In [ ]:
import numpy as np
import pandas as pd
import plotly.io as pio

import tsam
from tsam import ClusterConfig, Distribution
from tsam.plot import compare_series

pio.renderers.default = "notebook_connected"

raw = pd.read_csv("../../data/testdata.csv", index_col=0, parse_dates=True)
load = raw["Load"]
districts = pd.DataFrame({"District A": load, "District B": 0.6 * load.shift(2)})
data = districts.loc["2010-04-01":"2010-04-14"]
UNITS = {"District A": "MW", "District B": "MW"}
COLORS = {"original": "#000000"}


def aggregate(representation):
    return tsam.aggregate(
        data,
        n_clusters=2,
        period_duration="1D",
        cluster=ClusterConfig(method="hierarchical", representation=representation),
        preserve_column_means=False,
    )


def combined(frame):
    return frame.sum(axis=1).to_frame("Combined load")

Look at both districts over the two weeks:

In [ ]:
compare_series(
    {"original": data},
    units=UNITS,
    colors=COLORS,
    title="Two districts, 1 to 14 April 2010",
).show()

District B follows the same pattern as A, smaller and two hours later. The connection has to
carry their sum. Find its highest hour:

In [ ]:
total = combined(data)["Combined load"]
print(f"{total.max():.1f} MW on {total.idxmax():%d %B at %H:%M}")

In [ ]:
# Keeps the text true when the notebook runs in CI; hidden in the docs.
assert round(total.max(), 1) == 896.3
assert round(data.max().sum(), 1) == 900.0

896.3 MW, on 9 April at 12:30. The two districts never reach their own peaks, 562.5 and
337.5 MW, in the same hour, so the combined peak stays a little below their sum of 900 MW.

## 2  The default typical days

Reduce the two weeks with the default ordering and find the combined peak of the typical
days. `result.reconstructed` lays them back out over the two weeks, one copy for every day
they represent:

In [ ]:
independent = aggregate(Distribution())

peak = combined(independent.reconstructed)["Combined load"].max()
print(f"{peak:.1f} MW")

In [ ]:
# Keeps the text true when the notebook runs in CI; hidden in the docs.
assert round(peak, 1) == 879.5

17 MW too little: a connection sized on these typical days would be overloaded on 9 April.
Look at when each district peaks on the larger typical day, the one that represents 9 days:

In [ ]:
counts = independent.cluster_counts
typical = max(counts, key=counts.get)
independent.cluster_representatives.loc[typical].idxmax()

In [ ]:
# Keeps the text true when the notebook runs in CI; hidden in the docs.
assert counts[typical] == 9
hours = independent.cluster_representatives.loc[typical].idxmax()
assert hours["District B"] - hours["District A"] == 2

District A peaks at timestep 11, district B at 13. Each district keeps its own timing, so
their peaks stay two hours apart and never add up.

## 3  Share one order

A shared ordering gives both districts the same clock: the highest value of each district
lands in the same timestep. `assignment` is one of them ([part 4](04_ordering.ipynb) compares
all five orderings):

In [ ]:
shared = aggregate(Distribution(concurrency="assignment"))

peak = combined(shared.reconstructed)["Combined load"].max()
print(f"{peak:.1f} MW")
shared.cluster_representatives.loc[typical].idxmax()

In [ ]:
# Keeps the text true when the notebook runs in CI; hidden in the docs.
assert round(peak, 1) == 892.2
assert peak < total.max()
hours = shared.cluster_representatives.loc[typical].idxmax()
assert hours["District A"] == hours["District B"] == 12

Both peaks now sit at timestep 12, and the combined peak rises to 892.2 MW. It is still below
the real 896.3 MW: the highest value of each district on a typical day is an average of
several hours ([part 3](03_extremes.ipynb) shows why).

## 4  Keep the highest values, too

`preserve_minmax=True` keeps the real highest and lowest value of each district on every
typical day. Combine it with the shared order:

In [ ]:
conservative = aggregate(Distribution(concurrency="assignment", preserve_minmax=True))

peak = combined(conservative.reconstructed)["Combined load"].max()
print(f"{peak:.1f} MW")

900.0 MW, more than the real combined peak. Check it on each typical day, against the highest
combined hour of the days it represents:

In [ ]:
rows = {}
for cid, n_days in counts.items():
    member_days = np.flatnonzero(independent.cluster_assignments == cid)
    its_days = pd.concat([data.iloc[day * 24 : (day + 1) * 24] for day in member_days])
    rows[f"typical day for {n_days:.0f} days"] = {
        "its days": combined(its_days)["Combined load"].max(),
        "independent": combined(independent.cluster_representatives.loc[cid])[
            "Combined load"
        ].max(),
        "assignment + preserve_minmax": combined(
            conservative.cluster_representatives.loc[cid]
        )["Combined load"].max(),
    }
comparison = pd.DataFrame(rows).T
comparison.round(1)

In [ ]:
# Keeps the text true when the notebook runs in CI; hidden in the docs.
assert round(peak, 1) == 900.0
assert (comparison["assignment + preserve_minmax"] >= comparison["its days"]).all()
assert (comparison["independent"] < comparison["its days"]).all()

On both typical days, the conservative combined peak is at least as high as on any of its
days, and the default one is lower. That is no coincidence. Each typical day now holds the
highest load of district A and the highest load of district B in the same timestep, and no
real hour among its days can exceed their sum.

## 5  Choose the hour of the peak

Every shared ordering pairs the highest values with each other, so all of them give the same
combined peak. They differ in the timestep where they put it. `reference` uses the clock of
one attribute. Try it with each district:

In [ ]:
peaks = {}
for name, representation in {
    "assignment": Distribution(concurrency="assignment", preserve_minmax=True),
    "reference A": Distribution(
        concurrency="reference", reference_attribute="District A", preserve_minmax=True
    ),
    "reference B": Distribution(
        concurrency="reference", reference_attribute="District B", preserve_minmax=True
    ),
}.items():
    day = combined(aggregate(representation).cluster_representatives.loc[typical])
    peaks[name] = {
        "combined peak (MW)": day["Combined load"].max(),
        "timestep": day["Combined load"].idxmax(),
    }
pd.DataFrame.from_dict(peaks, orient="index").round(1)

In [ ]:
# Keeps the text true when the notebook runs in CI; hidden in the docs.
values = [p["combined peak (MW)"] for p in peaks.values()]
np.testing.assert_allclose(values, values[0])
assert round(values[0], 1) == 900.0
assert peaks["assignment"]["timestep"] == 12
assert peaks["reference A"]["timestep"] == 11
assert peaks["reference B"]["timestep"] == 13

The combined peak is 900 MW in all three. `reference` puts it where the named district peaks
on its own, at timestep 11 for A and 13 for B. `assignment` chooses timestep 12, between the
two. If the hour of the peak matters to your model, for example because grid fees depend on
the time of day, name the district whose timing decides it.

## What you've learned

* The default ordering keeps each district's own timing. Their peaks stay two hours apart,
  and the typical days underestimate the combined peak: 879.5 instead of 896.3 MW.
* A shared ordering puts the highest values of all attributes into the same timestep. With
  `preserve_minmax=True`, the combined peak of every typical day is at least as high as the
  highest combined hour of its days: a conservative estimate.
* All shared orderings give the same combined peak. `reference` lets you choose its hour by
  naming the attribute whose timing matters.

Where to go next:

* [Part 3: Keeping the extremes](03_extremes.ipynb) — keep the real highest and lowest values.
* [Part 4: Which values occur together](04_ordering.ipynb) — all five orderings, and what they
  do to realistic pairs of values.